In [ ]:
import os
import re
import csv
import json
import time
from groq import Groq
from datasets import load_dataset
from openai import OpenAI

In [ ]:
API_KEY = "Your key here"

router_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
groq_client = groq_client = Groq(api_key=API_KEY)

In [ ]:
##### MMLU PRO ####

def save_json(raw_text_data, saved_items, filename="data/mmlu_pro_raw_answers.json"):
    """
    Saves the full raw LLM responses to a JSON file.
    """
    correct_answers = {item["question_id"]: item["answer"] for item in saved_items}
    
    # Build the final output structure
    output_data = []
    for q_id, model_responses in raw_text_data.items():
        output_data.append({
            "question_id": q_id,
            "model_responses": model_responses, # This will now be the full answer_text
            "correct_response": correct_answers.get(q_id, "N/A")
        })
        
    # Write to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=4)



MODELS = [
    ("groq", "openai/gpt-oss-20b"),
    ("groq", "openai/gpt-oss-120b"),
    ("router","poolside/laguna-s-2.1:free"),
    ("router","nvidia/nemotron-3-super-120b-a12b:free"),
    ("router","microsoft/phi-4"),
    ("router","qwen/qwen3-8b"),
    ("router","mistralai/ministral-8b-2512"),
    ("router","meta-llama/llama-3.3-70b-instruct"),
    ("router","google/gemma-4-26b-a4b-it"),
    ("router","z-ai/glm-4.5-air"),
    ("router","nvidia/nemotron-3-nano-30b-a3b"),
    ("router","deepseek/deepseek-v3.2-exp")]


N_QUESTIONS = 200 
LETTERS = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]
REASONING_MODELS = ("gpt-oss", "deepseek", "laguna","nemotron","gemma","phi","glm")
HYBRID_MODELS = ("qwen3", "qwen2.5")

CSV_PATH = "data/MMLU_PRO_benchmark_results.csv"
QUESTIONS_JSON_PATH = "data/PRO_selected_questions.json"
ANS_PATH = "data/MMLU_PRO_sample_responses.json"

# get dataset
if os.path.exists(QUESTIONS_JSON_PATH):
    print(f"Loading cached dataset from {QUESTIONS_JSON_PATH}...")
    with open(QUESTIONS_JSON_PATH, "r", encoding="utf-8") as f:
        saved_items = json.load(f)
else:
    saved_items = []



if len(saved_items) < N_QUESTIONS:
    print(f"Expanding dataset cache from {len(saved_items)} to {N_QUESTIONS} questions...")
    raw_dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
    
    full_shuffled = raw_dataset.shuffle(seed=42)
    
    for i in range(len(saved_items), N_QUESTIONS):
        item = full_shuffled[i]
        saved_items.append({
            "question_id": f"Q{i+1}",
            "question": item["question"],
            "choices": item["options"], # MMLU-Pro uses 'options'
            "answer": item["answer"]    # MMLU-Pro 'answer' is already a letter (A-J)
        })
        
    with open(QUESTIONS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(saved_items, f, indent=2)
else:
    
    saved_items = saved_items[:N_QUESTIONS]


# Load existing results


existing_data = {item["question_id"]: {} for item in saved_items}
all_known_models = set(m[1] for m in MODELS)

if os.path.exists(CSV_PATH):
    print(f"Reading existing results from {CSV_PATH}...")
    with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames:
            
            for col in reader.fieldnames:
                if col not in ["question_id", "correct_label"]:
                    all_known_models.add(col)
            for row in reader:
                q_id = row["question_id"]
                if q_id in existing_data:
                    for model in all_known_models:
                        if model in row and row[model] != "":
                            existing_data[q_id][model] = row[model]


def format_question(item):
    choices = "\n".join(f"{LETTERS[idx]}. {c}" for idx, c in enumerate(item["choices"]))
    return (
        f"Answer the following multiple-choice question.\n"
        f"First, think step by step to solve the problem. "
        f"Then, you MUST end your response with the exact phrase: 'The answer is (X)' where X is the correct letter (A-J).\n\n"
        f"Question: {item['question']}\n{choices}\n\n"
    )

def extract_letter(text):
    if not text:
        return None
        
    match = re.search(r"answer is \(?([A-J])\)?", text, re.IGNORECASE)
    
    if match:
        return match.group(1).upper()
        
    # Optional fallback if it just spits out a single letter at the end
    fallback = re.search(r"\b([A-J])\b\.?$", text.strip())
    return fallback.group(1).upper() if fallback else None

def ask(provider, model, prompt, retries=3):
    client = groq_client if provider == "groq" else router_client
    for attempt in range(retries):
        try:
            kwargs = dict(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            if any(tag in model for tag in REASONING_MODELS):
                kwargs["max_tokens"] = 2048
                kwargs["reasoning_effort"] = "low"
            elif any(tag in model for tag in HYBRID_MODELS):
                kwargs["max_tokens"] = 2048  #1024
                kwargs["reasoning_effort"] = "minimal"
            else:
                kwargs["max_tokens"] = 2048
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limited on {model}, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    return None

def save_csv():
    # Sort headers 
    sorted_models = sorted(list(all_known_models))
    headers = ["question_id"] + sorted_models + ["correct_label"]
    
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for item in saved_items:
            q_id = item["question_id"]
            row = {"question_id": q_id, "correct_label": item["answer"]}
            # Populate predictions we have
            for model in sorted_models:
                row[model] = existing_data[q_id].get(model, "")
            writer.writerow(row)


#  RUN INFERENCE 
raw_text_data = {}
print("\nStarting evaluation loop...")
for idx, item in enumerate(saved_items):
    q_id = item["question_id"]
    prompt = format_question(item)  
    
    for provider, model in MODELS:
    
        saved_answer = existing_data[q_id].get(model)
        

        if saved_answer:# and saved_answer != "N/A":
            continue
  
            
        print(f"Fetching {q_id} for {model} ({provider})...")
        answer_text = ask(provider, model, prompt)
        predicted = extract_letter(answer_text or "")
        # 2. Save the COMPLETE raw text for your JSON
        #raw_text_data.setdefault(q_id, {})[model] = answer_text or "N/A"
        #save_json(raw_text_data, saved_items, "answers.json")
        # Save 
        existing_data[q_id][model] = predicted or "N/A"
        save_csv()
        
        print(f"  -> Predicted: {predicted} | Expected: {item['answer']} ") #| Raw: {answer_text} 
        time.sleep(2.5)